In [7]:
import requests

url = "http://10.8.4.19:8500/custOrders/createNewOrderId"
# payload = {
#     "FILIAL_ID": 17,
#     "CUSTOMER_ACCOUNT_ID": 700371,
#     "SALES_CHANNEL_ID": 1,
#     "EXTERNAL_ID": "EXT-0001",
#     "PRODUCT_OFFER_ID": 123456,
#     "ADDRESS": {
#         "STREET_ID": 123,
#         "HOUSE": 10,
#         "SUB_HOUSE": "A",
#         "FLAT": 45,
#         "SUB_FLAT": "2",
#         "ZIP_CODE": "050000"
#     },
#     "CUST_ORDER_ITEMS": [{
#         "EXTERNAL_ID": "EXT-0001-1",
#         "ORDER_NUM": 1,
#         "PO_COMPONENT_ID": -1,
#         "PRODUCT_OFFER_STRUCT_ID": 555,
#         "PRODUCT_NUM": "ACC-001",
#         "RESOURCE_SPEC_ID": 4444,
#         "SERVICE_COUNT": 1,
#         "PO_STRUCT_ELEMENTS": [{
#             "PO_STRUCT_ELEMENT_ID": 777,
#             "ACTION_DATE": "2025-10-02T09:00:00.000",
#             "SERVICE_COUNT": 1
#         }]
#     }]
# }

r = requests.post(url, json=payload, headers={"Content-Type": "application/json"})
print(r.status_code, r.text)


ConnectTimeout: HTTPConnectionPool(host='10.8.4.19', port=8500): Max retries exceeded with url: /custOrders/createNewOrderId (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x0000028E2CA03F10>, 'Connection to 10.8.4.19 timed out. (connect timeout=None)'))

In [24]:
import requests
url = "http://10.8.219.66:8501/rest/oapi/custOrders/createNewOrderId"
data= {
    "AUTHORIZATION": "Api-Key ItU2iuhEiYKeYl7ZVekueetugTU79mge"
}
r = requests.post(url,  headers=data)
print(r.status_code, r.text)

200 {"ERRORMESSAGE":"","DATA":{"id":138190},"ERRORCODE":0}


In [27]:
import requests
url = "http://10.8.4.233:80/rest/oapi/custOrders/createNewOrderId"
data= {
    "AUTHORIZATION": "Api-Key ItU2iuhEiYKeYl7ZVekueetugTU79mge"
}
r = requests.post(url,  headers=data)
print(r.status_code, r.text)


403 {"Message":"Forbidden","Type":"Application"}


In [7]:
import os
import json
import time
from typing import Dict, Any, Optional, Tuple

import requests
from requests.adapters import HTTPAdapter, Retry

# ----------------------------
# Configuration
# ----------------------------
BASE_URL = "http://10.8.219.66:8501"
API_KEY = "ItU2iuhEiYKeYl7ZVekueetugTU79mge"  # <- you can also use os.environ.get("ORDER_API_KEY")

TIMEOUT = (5, 30)  # (connect, read) seconds

HEADERS = {
    "Content-Type": "application/json",
    "AUTHORIZATION": API_KEY,  # per spec, no 'Bearer' prefix
}

# Robust session with retries for transient errors
session = requests.Session()
retries = Retry(
    total=3,
    backoff_factor=0.4,
    status_forcelist=(500, 502, 503, 504),
    allowed_methods=frozenset(["GET", "POST", "PUT", "DELETE", "PATCH"]),
)
session.mount("http://", HTTPAdapter(max_retries=retries))
session.mount("https://", HTTPAdapter(max_retries=retries))


def _pretty(d: Any) -> str:
    try:
        return json.dumps(d, ensure_ascii=False, indent=2, sort_keys=True)
    except Exception:
        return str(d)


def _handle_response(resp: requests.Response) -> Tuple[int, str, Dict[str, Any]]:
    """
    Returns (err_code, err_msg, payload_dict)
    """
    try:
        payload = resp.json()
    except ValueError:
        # Non-JSON response
        payload = {"raw": resp.text}

    err_code = None
    err_msg = ""
    if isinstance(payload, dict):
        # Common fields in this API spec
        err_code = payload.get("ERR_CODE")
        err_msg = payload.get("ERR_MSG") or payload.get("message") or ""
    # Fallback when API didn't include standard fields
    if err_code is None:
        err_code = 0 if resp.ok else resp.status_code
        if not err_msg:
            err_msg = f"HTTP {resp.status_code}"

    return err_code, err_msg, payload


In [8]:


def create_work_order(body: Dict[str, Any]) -> Dict[str, Any]:
    """
    POST /rest/oapi/order/workOrder
    Required (per spec): filialId, customerAccountId, externalId, productOfferId, address(streetId, houseNo, flatNo).
    Optionally include poStructElements and installDate.
    """
    url = f"{BASE_URL}/rest/oapi/order/workOrder"
    resp = session.post(url, headers=HEADERS, data=json.dumps(body), timeout=TIMEOUT)
    err_code, err_msg, payload = _handle_response(resp)

    print("== Create Work Order ==")
    print("Request Body:")
    print(_pretty(body))
    print("\nResponse:")
    print(_pretty(payload))

    if err_code != 0:
        raise RuntimeError(f"API error (ERR_CODE={err_code}): {err_msg}")

    return payload

In [9]:
def create_new_order(body: Dict[str, Any]) -> Dict[str, Any]:
    """
    POST /rest/oapi/order/create_new
    Required (per spec): filialId, customerAccountId, salesChannelId, externalId,
    productOfferId, address(streetId, houseNo, flatNo), custOrderItems[*] with (externalId, actionId, productOfferStructId).
    """
    url = f"{BASE_URL}/rest/oapi/order/create_new"
    resp = session.post(url, headers=HEADERS, data=json.dumps(body), timeout=TIMEOUT)
    err_code, err_msg, payload = _handle_response(resp)

    print("== Create New Product Offer Order ==")
    print("Request Body:")
    print(_pretty(body))
    print("\nResponse:")
    print(_pretty(payload))

    if err_code != 0:
        raise RuntimeError(f"API error (ERR_CODE={err_code}): {err_msg}")

    return payload

In [10]:
# ----------------------------
# Example payloads (EDIT THESE)
# ----------------------------

# EXAMPLE_CREATE_NEW_PAYLOAD = {
#     "filialId": 1,                     # required
#     "customerAccountId": 700371,       # required
#     "salesChannelId": 10,              # required
#     "externalId": "EXT-12345",         # required (your external order id)
#     "productOfferId": 1234,            # required (base/bundle product offer)
#     "address": {                       # required
#         "streetId": 555,               # required in spec
#         "houseNo": "12",               # required
#         "houseFraction": None,
#         "flatNo": "45",                # required
#         "flatFraction": None,
#         "zip": "050000"
#     },
#     "custOrderItems": [                # required
#         {
#             "externalId": "EXT-ITEM-1",    # required
#             "actionId": 1,                 # required: 1=install, 2=change, 3=disconnect
#             "productOfferStructId": 9876,  # required
#             "productInstanceId": None,
#             "productNum": None,
#             "resourceSpecId": None
#         }
#     ],
#     "comments": "Created via Jupyter"
# }

EXAMPLE_WORK_ORDER_PAYLOAD = {
    "orderId": 137785,                   # optional (use if updating existing)
    "filialId": 1,                     # required
    "customerAccountId": 700371,       # required
    "externalId": "EXT-WO-12345",      # required
    "productOfferId": 1234,            # required
    "comments": "Technician visit needed",
    "address": {                       # required
        "streetId": 555,               # required
        "houseNo": "12",               # required
        "houseFraction": None,
        "flatNo": "45",                # required
        "flatFraction": None,
        "zip": "050000"
    },
    "poStructElements": [
        {
            "externalId": "EXT-WO-ITEM-1",
            "productNum": None
        }
    ],
    "installDate": "2025-10-20 10:00:00"  # optional, if scheduling a visit
}

# ----------------------------
# How to run:
# ----------------------------
# 1) For a new product offer order:
# result = create_new_order(EXAMPLE_CREATE_NEW_PAYLOAD)
#
# 2) For a work order:
result = create_work_order(EXAMPLE_WORK_ORDER_PAYLOAD)
#
print("ORDER RESULT:", _pretty(result))

== Create Work Order ==
Request Body:
{
  "address": {
    "flatFraction": null,
    "flatNo": "45",
    "houseFraction": null,
    "houseNo": "12",
    "streetId": 555,
    "zip": "050000"
  },
  "comments": "Technician visit needed",
  "customerAccountId": 700371,
  "externalId": "EXT-WO-12345",
  "filialId": 1,
  "installDate": "2025-10-20 10:00:00",
  "orderId": 137785,
  "poStructElements": [
    {
      "externalId": "EXT-WO-ITEM-1",
      "productNum": null
    }
  ],
  "productOfferId": 1234
}

Response:
{
  "errCode": -1,
  "errMessage": "Отсутствует обязательный параметр [custOrderItem] в данных заказа | "
}
ORDER RESULT: {
  "errCode": -1,
  "errMessage": "Отсутствует обязательный параметр [custOrderItem] в данных заказа | "
}


In [33]:
FIXED_WORK_ORDER_PAYLOAD = {
    "orderId": 137785,            # keep if you’re updating an existing order; else None
    "filialId": 1,
    "customerAccountId": 700371,
    "externalId": "EXT-WO-12345",
    "productOfferId": 1234,
    "comments": "Technician visit needed",
    "address": {
        "streetId": 555,
        "house": "12",
        "houseFraction": None,
        "flatNo": "45",
        "flatFraction": None,
        "zipCode": "050000"
    },
    # ✨ REQUIRED for this endpoint (singular): custOrderItem
    "custOrderItem": {
        "externalId": "EXT-WO-ITEM-1",
        "actionId": 1,                 # 1=install, 2=change, 3=disconnect (per spec)
        "productOfferStructId": 9876,  # PO struct element you’re acting on
        "productInstanceId": None,
        "productNum": "123",
        "resourceSpecId": None,
        "oneChargeElement":[
           2345324
        ]
        
    },
    # Optional: you can still include poStructElements if your backend uses it

    "installDate": "2025-10-20 10:00:00"
}
result = create_work_order(FIXED_WORK_ORDER_PAYLOAD)
print(_pretty(result))


== Create Work Order ==
Request Body:
{
  "address": {
    "flatFraction": null,
    "flatNo": "45",
    "house": "12",
    "houseFraction": null,
    "streetId": 555,
    "zipCode": "050000"
  },
  "comments": "Technician visit needed",
  "custOrderItem": {
    "actionId": 1,
    "externalId": "EXT-WO-ITEM-1",
    "oneChargeElement": [
      2345324
    ],
    "productInstanceId": null,
    "productNum": "123",
    "productOfferStructId": 9876,
    "resourceSpecId": null
  },
  "customerAccountId": 700371,
  "externalId": "EXT-WO-12345",
  "filialId": 1,
  "installDate": "2025-10-20 10:00:00",
  "orderId": 137785,
  "productOfferId": 1234
}

Response:
{
  "errCode": -1,
  "errMessage": "Неверный формат массива элементов | "
}
{
  "errCode": -1,
  "errMessage": "Неверный формат массива элементов | "
}


In [ ]:
import requests
import json
from datetime import datetime

# Configuration
API_KEY = "ItU2iuhEiYKeYl7ZVekueetugTU79mge"
BASE_URL = "http://10.8.219.66:8501"
CUSTOMER_ACCOUNT_ID = 12319534

# Headers
headers = {
    "Content-Type": "application/json",
    "AUTHORIZATION": API_KEY
}

# Order payload for user 12319534
order_payload = {
    "FILIAL_ID": 17,  # Default filial ID from your app
    "CUSTOMER_ACCOUNT_ID": CUSTOMER_ACCOUNT_ID,
    "SALES_CHANNEL_ID": 1,
    "EXTERNAL_ID": f"EXT-{CUSTOMER_ACCOUNT_ID}-{int(datetime.now().timestamp())}",
    "PRODUCT_OFFER_ID": 123456,  # You'll need to specify the actual product offer ID
    "ADDRESS": {
        "STREET_ID": 123,
        "HOUSE": 1,
        "SUB_HOUSE": None,
        "FLAT": 1,
        "SUB_FLAT": None,
        "ZIP_CODE": "050000"
    },
    "CUST_ORDER_ITEMS": [{
        "EXTERNAL_ID": f"EXT-{CUSTOMER_ACCOUNT_ID}-ITEM-1",
        "ORDER_NUM": 1,
        "PO_COMPONENT_ID": -1,
        "PRODUCT_OFFER_STRUCT_ID": 555,  # You'll need to specify the actual struct ID
        "PRODUCT_NUM": "ACC-001",  # You'll need to specify the actual product number
        "RESOURCE_SPEC_ID": 4444,  # You'll need to specify the actual resource spec ID
        "SERVICE_COUNT": 1,
        "PO_STRUCT_ELEMENTS": [{
            "PO_STRUCT_ELEMENT_ID": 777,  # You'll need to specify the actual element ID
            "ACTION_DATE": datetime.now().strftime("%Y-%m-%dT%H:%M:%S.000"),
            "SERVICE_COUNT": 1
        }]
    }]
}

# Make the API call
try:
    response = requests.post(
        f"{BASE_URL}/rest/oapi/order/create_new",
        json=order_payload,
        headers=headers,
        timeout=30
    )
    
    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.text}")
    
    if response.status_code == 200:
        result = response.json()
        if result.get("ERRORCODE") == 0:
            print(f"✅ Order created successfully!")
            print(f"Order ID: {result.get('DATA', {}).get('id')}")
        else:
            print(f"❌ API Error: {result.get('ERRORMESSAGE')}")
    else:
        print(f"❌ HTTP Error: {response.status_code}")
        
except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

In [ ]:
print('as')